# MàthemaSharp.TimeSeries - consolidation

## Usage

Examples and checks in T on the six formats. Prerequisites: Release build and a JupyterSharp kernel running on the .NET 11 runtime. The comparisons are on small cases: they do not certify all the APIs or general accuracy. No Charts or NDSharp dependency.



In [ ]:
#load "../shared/MathemaSharp.Numerics.csx"
using System.Numerics;
using MathemaSharp.TimeSeries;
void Verify<T>() where T : IFloatingPointIeee754<T>, IMinMaxValue<T>
{
    T N(int n) => T.CreateChecked(n);
    var eps = T.BitIncrement(T.One)-T.One;
    var loess = Stl.Loess<T>([N(1),N(3),N(5),N(7),N(9)],[],5,N(2));
    if (T.Abs(loess-N(5)) > N(64)*eps*N(5)) throw new Exception("LOESS");
    var walk = Simulation.RandomWalk<T>(new Random(42),3,T.Zero,T.Zero,T.One,_ => T.One);
    if (walk[2] != N(3)) throw new Exception("Innovations");
    var fit = Garch11.FitVarianceTargeting<T>([T.One,-T.One,N(2),-N(2)],8,1);
    if (fit.Persistence >= T.One || !T.IsFinite(fit.Omega)) throw new Exception("GARCH");
    if (SpecialFunctions.RegularizedGammaQ(T.One,T.One) != T.Exp(-T.One)) throw new Exception("Gamma");
    Console.WriteLine($"{typeof(T).Name}: LOESS=OK innovations=OK GARCH=OK Q(1,x)=OK");
}
Verify<Half>();
Verify<float>();
Verify<double>();
Verify<Decimal32>();
Verify<Decimal64>();
Verify<Decimal128>();



Half: LOESS=OK innovations=OK GARCH=OK Q(1,x)=OK
Single: LOESS=OK innovations=OK GARCH=OK Q(1,x)=OK
Double: LOESS=OK innovations=OK GARCH=OK Q(1,x)=OK
Decimal32: LOESS=OK innovations=OK GARCH=OK Q(1,x)=OK
Decimal64: LOESS=OK innovations=OK GARCH=OK Q(1,x)=OK
Decimal128: LOESS=OK innovations=OK GARCH=OK Q(1,x)=OK

## Reference - APIs used and related families

| API | Constraint / limit |
|---|---|
| Stl.Loess | IEEE; centred abscissae |
| Garch11.FitVarianceTargeting | IEEE; grid estimator, global maximum not guaranteed |
| Simulation + innovationSource | Innovations in T; distribution left to the caller |
| Simulation default | Box-Muller double converted to T |
| SpecialFunctions | IEEE; Lanczos with limited precision, not 34 digits |

This table does not replace the full Reference. [Usage](../../USAGE.md), [API inventory and limits](../../ARCHITECTURE.md), [roadmap](../../ROADMAP.md). The saved outputs testify to the execution of these cells, not of the previous notebooks.

